# Gait-Based Subject Identification: Windowed Features + 1D CNN

**Problem.** Identify which of the subjects produced an accelerometer recording (right trouser pocket, ~100 Hz).

**Why this notebook differs from a row-level approach.** A single `(X, Y, Z)` sample carries almost no information about *who* is walking. Gait is a pattern over time (step rhythm, swing shape, stride regularity), so this notebook:

1. Cuts the signal into **2-second windows** (200 samples, 50% overlap), never crossing a time gap or a G/R label boundary.
2. Evaluates with a **cross-session protocol**: train on one session, test on the other session of the *same* subjects. This is the honest number. A random window split is also shown, **for reference only**, to demonstrate how much it inflates accuracy.
3. Compares tree models on hand-crafted features (full vs. orientation-invariant) with a **1D CNN on raw windows**.
4. Reports both **window-level** accuracy and **vote accuracy** (averaging predictions over K consecutive windows).

**Protocol summary**

| Item | Choice |
|---|---|
| Evaluation subjects | the subjects that have **both** Session 1 and Session 2 |
| Train | evaluation subjects, first 80% (in time) of the training session |
| Validation | last 20% (in time) of the training session, with a purge gap so windows don't overlap |
| Test | the *other* session of the same subjects |
| Extra training data | single-session subjects can be extra classes only when their recording is in the training session; predictions are restricted to evaluation subjects |

## 1. Imports and configuration

In [ ]:
import os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from numpy.lib.stride_tricks import sliding_window_view  # not required, kept for experimentation
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier

warnings.filterwarnings("ignore")

# ---------------- configuration ----------------
SEED = 42
FS = 100                 # sampling rate (Hz)
WIN = 200                # window length in samples (2 s)
STEP = 100               # hop in samples (50% overlap)
MAX_GAP_S = 0.05         # a timestamp jump larger than this breaks a window

VAL_FRACTION = 0.20      # last part (in time) of the training session used for validation
USE_EXTRA_SUBJECTS = True    # add single-session subjects to training as extra classes
DIRECTIONS = [(1, 2), (2, 1)]  # (train_session, test_session)
VOTE_K = 10              # number of consecutive windows averaged for "vote accuracy"

# tree models
N_TREES = 200
N_JOBS = -1

# CNN
REMOVE_MEAN = True       # subtract each window's mean (removes gravity / orientation offset)
EPOCHS = 40
BATCH = 256
LR = 2e-3
WEIGHT_DECAY = 1e-2
USE_EARLY_STOPPING = False   # validation is the same session as training (~99% acc), so it cannot pick a good epoch
PATIENCE = 8             # only used if USE_EARLY_STOPPING = True
ROT_DEG = 20.0           # max random rotation (augmentation)
SCALE_JIT = 0.10         # +/- random amplitude scaling (augmentation)
NOISE_STD = 0.02         # gaussian jitter (augmentation, in input units)

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

## 2. Load the data

In [ ]:
DATA_FILE = "right_pocket_sessions_1_2_combined.csv"
search = [
    Path.cwd() / DATA_FILE,
    Path.cwd() / "Combined_RightPocket" / DATA_FILE,
    Path("/mnt/user-data/uploads") / DATA_FILE,
]
DATA_PATH = next((p for p in search if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Could not find {DATA_FILE}. Set DATA_PATH manually.")

df = pd.read_csv(
    DATA_PATH,
    dtype={"X": "float32", "Y": "float32", "Z": "float32",
           "Subject": "int16", "Session": "int8", "Label": "category"},
)
print("File   :", DATA_PATH)
print("Rows   :", f"{len(df):,}")
print("Columns:", list(df.columns))
print("NaNs   :", int(df[["X", "Y", "Z", "Timestamp"]].isna().sum().sum()))
print("Subjects:", df.Subject.nunique(), "| Sessions:", sorted(df.Session.unique()), "| Labels:", list(df.Label.cat.categories))

sess_per_subject = df.groupby("Subject").Session.nunique()
print("Subjects with both sessions:", int((sess_per_subject == 2).sum()),
      "| single-session only:", int((sess_per_subject == 1).sum()))

## 3. Windowing

Windows are built **inside each (Subject, Session, Label) block**, so a window never mixes two recordings or crosses the G→R boundary.
Any window containing a timestamp gap larger than `MAX_GAP_S` is dropped (the file contains ~30 s gaps between recorded chunks).
Each window keeps its chronological position (`pos`) inside its subject-session, which is used for the time-based validation split and for vote accuracy.

In [ ]:
def build_windows(frame):
    xs, metas = [], []
    for (sub, ses, lab), g in frame.groupby(["Subject", "Session", "Label"], sort=False, observed=True):
        a = g[["X", "Y", "Z"]].to_numpy(np.float32)
        t = g["Timestamp"].to_numpy()
        n = len(a)
        if n < WIN:
            continue
        gap = np.zeros(n, dtype=np.int32)
        gap[1:] = (np.diff(t) > MAX_GAP_S)
        cs = np.cumsum(gap)
        starts = np.arange(0, n - WIN + 1, STEP)
        ok = (cs[starts + WIN - 1] - cs[starts]) == 0      # no gap inside the window
        starts = starts[ok]
        if len(starts) == 0:
            continue
        idx = starts[:, None] + np.arange(WIN)[None, :]
        xs.append(a[idx])                                  # (n_windows, WIN, 3)
        metas.append(pd.DataFrame({"subject": int(sub), "session": int(ses),
                                   "label": str(lab), "start": starts}))
    Xw = np.concatenate(xs).astype(np.float32)
    meta = pd.concat(metas, ignore_index=True)
    # chronological position inside each subject-session (G block first, then R)
    meta["pos"] = meta.groupby(["subject", "session"]).cumcount()
    return Xw, meta

t0 = time.time()
Xw, meta = build_windows(df)
del df
print(f"Windows: {Xw.shape}  ({time.time() - t0:.1f}s)")
print(meta.groupby("session").size().rename("windows per session").to_string())

# subject label encoding over ALL subjects (needed if extra subjects are used in training)
le = LabelEncoder().fit(meta.subject.values)
y_all = le.transform(meta.subject.values)
N_CLASSES = len(le.classes_)

both_sessions = meta.groupby("subject").session.nunique()
EVAL_SUBJECTS = sorted(both_sessions[both_sessions == 2].index.tolist())
EVAL_CLS = le.transform(EVAL_SUBJECTS)
print(f"Evaluation subjects (both sessions): {len(EVAL_SUBJECTS)} | chance = {1 / len(EVAL_SUBJECTS):.3%}")

## 4. Train / validation / test split

For a direction `(train_session, test_session)`:

* **Test**: all windows of the evaluation subjects from `test_session`.
* **Train / validation**: evaluation-subject windows from `train_session` are split **by time** (first 80% / last 20%), with a purge of `WIN // STEP` windows so samples do not overlap. The held-out validation tail may be added back for the final refit after model selection.
* The validation set is from the same session as training, so it cannot measure session generalization; the **other session** is the outer test set.
* **Extra** (optional): single-session subjects are extra training classes only when their recording is from `train_session`. No examples from `test_session` are used in training.

In [ ]:
def make_split(meta, train_ses, test_ses):
    is_eval = meta.subject.isin(EVAL_SUBJECTS).values
    train_session = meta.session.values == train_ses
    test_mask = is_eval & (meta.session.values == test_ses)

    train_mask = np.zeros(len(meta), dtype=bool)
    val_mask = np.zeros(len(meta), dtype=bool)
    purge = WIN // STEP

    cand = meta[is_eval & train_session]
    for sub, g in cand.groupby("subject"):
        idx = g.sort_values("pos").index.to_numpy()
        cut = int(len(idx) * (1 - VAL_FRACTION))
        train_mask[idx[:cut]] = True
        val_mask[idx[cut + purge:]] = True

    # Extra classes must come only from the training session; using the other
    # session would leak held-out-session conditions into model fitting.
    if USE_EXTRA_SUBJECTS:
        train_mask |= (~is_eval) & train_session

    # sanity checks: no overlap between the sets
    assert not (train_mask & val_mask).any()
    assert not (train_mask & test_mask).any()
    assert not (val_mask & test_mask).any()
    assert (meta.session.values[train_mask] == train_ses).all()
    return train_mask, val_mask, test_mask


for tr, te in DIRECTIONS:
    a, b, c = make_split(meta, tr, te)
    print(f"Session {tr} -> {te}: train {a.sum():,} | val {b.sum():,} | test {c.sum():,}")

## 5. Hand-crafted features

Two feature sets:

* **invariant**: depends only on the acceleration *magnitude* and on rotation-invariant quantities (eigenvalues of the 3x3 covariance of the mean-removed signal). Phone orientation in the pocket matters much less.
* **full**: invariant + per-axis mean/std/min/max and axis correlations (these encode orientation, which can help *or* hurt across sessions).

Magnitude features include statistics, a normalised FFT spectrum (0.5 Hz resolution up to 20 Hz), and autocorrelation values (stride regularity).

In [ ]:
def extract_features(Xw, full=True):
    N, W, _ = Xw.shape
    X = Xw.astype(np.float32)
    mag = np.linalg.norm(X, axis=2)                         # (N, W)
    mc = mag - mag.mean(1, keepdims=True)
    sd = mc.std(1) + 1e-8

    feats = []
    # magnitude statistics
    feats += [mag.mean(1), mag.std(1), mag.min(1), mag.max(1)]
    feats += list(np.percentile(mag, [10, 25, 50, 75, 90], axis=1))
    feats += [(mc ** 3).mean(1) / sd ** 3, (mc ** 4).mean(1) / sd ** 4]   # skewness, kurtosis
    feats += [np.abs(np.diff(mag, axis=1)).mean(1)]                         # mean absolute jerk-like term

    # spectrum of magnitude (Hann window), bins 1..40 -> 0.5..20 Hz
    spec = np.abs(np.fft.rfft(mc * np.hanning(W)[None, :], axis=1))[:, 1:41]
    energy = spec.sum(1) + 1e-8
    spec_n = spec / energy[:, None]
    feats += [energy, (np.argmax(spec, axis=1) + 1) * FS / W]               # energy, dominant freq (Hz)
    feats += list(spec_n.T)

    # autocorrelation of magnitude
    ft = np.fft.rfft(mc, n=2 * W, axis=1)
    ac = np.fft.irfft(ft * np.conj(ft), axis=1)[:, :W]
    ac = ac / (ac[:, :1] + 1e-8)
    feats += [ac[:, l] for l in range(10, 160, 10)]
    lo, hi = 40, 160
    feats += [ac[:, lo:hi].max(1), np.argmax(ac[:, lo:hi], axis=1) + lo]    # stride regularity and lag

    # rotation-invariant covariance eigenvalues of mean-removed xyz
    Xc = X - X.mean(1, keepdims=True)
    C = np.einsum("nwi,nwj->nij", Xc, Xc) / W
    ev = np.linalg.eigvalsh(C)                                              # ascending
    feats += [ev[:, 0], ev[:, 1], ev[:, 2]]

    if full:
        feats += list(X.mean(1).T) + list(X.std(1).T) + list(X.min(1).T) + list(X.max(1).T)
        sdx = np.sqrt(np.einsum("nii->ni", C)) + 1e-8
        for i, j in [(0, 1), (0, 2), (1, 2)]:
            feats += [C[:, i, j] / (sdx[:, i] * sdx[:, j])]

    return np.stack(feats, axis=1).astype(np.float32)

t0 = time.time()
F_full = extract_features(Xw, full=True)
F_inv = extract_features(Xw, full=False)
print("Features  full:", F_full.shape, "| invariant:", F_inv.shape, f"({time.time() - t0:.1f}s)")

## 6. Evaluation helpers

* `to_full_probs` maps a model's probabilities onto all classes, then **restricts** them to the evaluation subjects.
* `window_acc` is the accuracy per window.
* `vote_acc` averages the probabilities of `VOTE_K` consecutive windows of the same subject (about `VOTE_K` seconds of walking) before predicting.

In [ ]:
def restrict(P_all):
    # keep only evaluation-subject columns
    return P_all[:, EVAL_CLS]

def to_full_probs(proba, classes_, n_total):
    P = np.zeros((proba.shape[0], n_total), dtype=np.float32)
    P[:, classes_] = proba
    return P

def window_acc(P_eval, y_true):
    pred = EVAL_CLS[np.argmax(P_eval, axis=1)]
    return float((pred == y_true).mean())

def vote_acc(P_eval, y_true, meta_part, k=VOTE_K):
    key = meta_part.subject.astype(str).values + "_" + meta_part.session.astype(str).values + "_" + (meta_part.pos.values // k).astype(str)
    codes, _ = pd.factorize(key)
    Pg = pd.DataFrame(P_eval).groupby(codes).mean().to_numpy()
    yg = pd.Series(y_true).groupby(codes).first().to_numpy()
    pred = EVAL_CLS[np.argmax(Pg, axis=1)]
    return float((pred == yg).mean())

results = []   # rows: direction, model, window_acc, vote_acc

def log_result(direction, model, P_eval, y_true, meta_part):
    w, v = window_acc(P_eval, y_true), vote_acc(P_eval, y_true, meta_part)
    results.append({"direction": direction, "model": model, "window_acc": w, "vote_acc": v})
    print(f"  {model:<34s} window acc {w:6.2%} | vote({VOTE_K}) acc {v:6.2%}")

## 7. Machine-learning baselines (cross-session)

In [ ]:
def fit_tree(model, Xtr, ytr, Xte):
    model.fit(Xtr, ytr)
    return to_full_probs(model.predict_proba(Xte), model.classes_, N_CLASSES)

for tr, te in DIRECTIONS:
    direction = f"S{tr}->S{te}"
    train_mask, val_mask, test_mask = make_split(meta, tr, te)
    y_tr, y_te = y_all[train_mask], y_all[test_mask]
    meta_te = meta[test_mask]
    print(f"\n=== {direction} | train {train_mask.sum():,} | test {test_mask.sum():,} ===")

    for fname, FM in [("invariant", F_inv), ("full", F_full)]:
        et = ExtraTreesClassifier(n_estimators=N_TREES, n_jobs=N_JOBS, random_state=SEED)
        P = restrict(fit_tree(et, FM[train_mask], y_tr, FM[test_mask]))
        log_result(direction, f"ExtraTrees ({fname} feats)", P, y_te, meta_te)

    rf = RandomForestClassifier(n_estimators=N_TREES, n_jobs=N_JOBS, random_state=SEED)
    P = restrict(fit_tree(rf, F_full[train_mask], y_tr, F_full[test_mask]))
    log_result(direction, "RandomForest (full feats)", P, y_te, meta_te)

### Reference only: the leaky random window split

Overlapping neighbouring windows land in both train and test, so the model can memorise a recording instead of learning gait. This number is **not** a measure of generalisation.

In [ ]:
from sklearn.model_selection import train_test_split

mask_eval = meta.subject.isin(EVAL_SUBJECTS).values
idx_eval = np.where(mask_eval)[0]
tr_i, te_i = train_test_split(idx_eval, test_size=0.2, stratify=y_all[idx_eval], random_state=SEED)

et = ExtraTreesClassifier(n_estimators=N_TREES, n_jobs=N_JOBS, random_state=SEED)
et.fit(F_full[tr_i], y_all[tr_i])
leaky = float((et.predict(F_full[te_i]) == y_all[te_i]).mean())
print(f"Random window split (LEAKY, reference only): {leaky:.2%}")

## 8. 1D CNN on raw windows

**Input channels:** X, Y, Z and the magnitude (4 channels x 200 samples). If `REMOVE_MEAN` is on, each window's per-channel mean is subtracted (removes the gravity/orientation offset), then channels are standardised with training-set statistics.

**Augmentation (training only, on the GPU):** random small 3D rotation (simulates the phone sitting differently in the pocket), random amplitude scaling and Gaussian jitter.

**Training:** AdamW + one-cycle LR schedule, label smoothing, fixed number of epochs; the final-epoch weights are used.

**About validation:** the validation windows come from the *same session* as the training windows (the only session available before testing), so validation accuracy is expected to be ~99% and says little about cross-session performance. It is kept as a sanity check (is the model learning at all?). Early stopping on it is available via `USE_EARLY_STOPPING` but is off by default for that reason.

In [ ]:
def to_channels(xyz, remove_mean=REMOVE_MEAN):
    # xyz: (B, W, 3) float tensor -> (B, 4, W)
    mag = torch.linalg.norm(xyz, dim=2, keepdim=True)
    x = torch.cat([xyz, mag], dim=2)
    if remove_mean:
        x = x - x.mean(dim=1, keepdim=True)
    return x.transpose(1, 2).contiguous()

def random_rotation(B, max_deg, device):
    axis = torch.randn(B, 3, device=device)
    axis = axis / axis.norm(dim=1, keepdim=True).clamp_min(1e-8)
    ang = (torch.rand(B, device=device) * 2 - 1) * np.deg2rad(max_deg)
    K = torch.zeros(B, 3, 3, device=device)
    K[:, 0, 1], K[:, 0, 2] = -axis[:, 2], axis[:, 1]
    K[:, 1, 0], K[:, 1, 2] = axis[:, 2], -axis[:, 0]
    K[:, 2, 0], K[:, 2, 1] = -axis[:, 1], axis[:, 0]
    I = torch.eye(3, device=device).expand(B, 3, 3)
    s, c = torch.sin(ang)[:, None, None], torch.cos(ang)[:, None, None]
    return I + s * K + (1 - c) * (K @ K)

def augment(xyz):
    B = xyz.shape[0]
    R = random_rotation(B, ROT_DEG, xyz.device)
    xyz = torch.bmm(xyz, R.transpose(1, 2))
    scale = 1 + (torch.rand(B, 1, 1, device=xyz.device) * 2 - 1) * SCALE_JIT
    xyz = xyz * scale
    return xyz + torch.randn_like(xyz) * NOISE_STD


def conv_block(cin, cout, k):
    return nn.Sequential(nn.Conv1d(cin, cout, k, padding=k // 2, bias=False),
                         nn.BatchNorm1d(cout), nn.ReLU(inplace=True))

class GaitCNN(nn.Module):
    def __init__(self, in_ch, n_classes, p=0.3):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(in_ch, 32, 7),
            conv_block(32, 64, 5), nn.MaxPool1d(2),
            conv_block(64, 128, 5), nn.MaxPool1d(2),
            conv_block(128, 128, 3), nn.MaxPool1d(2),
            conv_block(128, 128, 3),
        )
        self.head = nn.Sequential(nn.Dropout(p), nn.Linear(256, n_classes))

    def forward(self, x):
        h = self.features(x)
        h = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)   # global avg + max pooling
        return self.head(h)

print("Parameters:", f"{sum(p.numel() for p in GaitCNN(4, N_CLASSES).parameters()):,}")

In [ ]:
@torch.no_grad()
def predict_probs(model, xyz_t, mu, sd, batch=2048):
    model.eval()
    out = []
    for i in range(0, len(xyz_t), batch):
        xb = (to_channels(xyz_t[i:i + batch]) - mu) / sd
        out.append(torch.softmax(model(xb), dim=1).cpu())
    return torch.cat(out).numpy()

def train_cnn(train_mask, val_mask, verbose=True):
    xyz_tr = torch.from_numpy(Xw[train_mask]).to(DEVICE)
    y_tr = torch.from_numpy(y_all[train_mask]).long().to(DEVICE)
    xyz_val = torch.from_numpy(Xw[val_mask]).to(DEVICE)
    y_val = y_all[val_mask]

    # channel statistics from the training set only
    ch = to_channels(xyz_tr)
    mu = ch.mean(dim=(0, 2), keepdim=True)
    sd = ch.std(dim=(0, 2), keepdim=True).clamp_min(1e-6)
    del ch

    model = GaitCNN(4, N_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(xyz_tr) / BATCH))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=steps_per_epoch)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_acc, best_state, bad = -1.0, None, 0
    hist = {"loss": [], "val_acc": []}
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(xyz_tr), device=DEVICE)
        total, n = 0.0, 0
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            xb = (to_channels(augment(xyz_tr[b])) - mu) / sd
            loss = loss_fn(model(xb), y_tr[b])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            total += loss.item() * len(b)
            n += len(b)

        P_val = restrict(predict_probs(model, xyz_val, mu, sd))
        val_acc = window_acc(P_val, y_val)
        hist["loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        if verbose:
            print(f"  epoch {epoch + 1:02d}/{EPOCHS}  loss {total / n:.4f}  val acc {val_acc:.2%}")

        if val_acc > best_acc:
            best_acc, bad = val_acc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if USE_EARLY_STOPPING and bad >= PATIENCE:
                if verbose:
                    print(f"  early stop (best val acc {best_acc:.2%})")
                break

    if USE_EARLY_STOPPING:
        model.load_state_dict(best_state)   # otherwise keep the final-epoch weights (end of the LR schedule)
    return model, mu, sd, hist

In [ ]:
cnn_hist = {}
for tr, te in DIRECTIONS:
    direction = f"S{tr}->S{te}"
    train_mask, val_mask, test_mask = make_split(meta, tr, te)
    print(f"\n=== CNN {direction} | train {train_mask.sum():,} | val {val_mask.sum():,} | test {test_mask.sum():,} ===")
    t0 = time.time()
    model, mu, sd, hist = train_cnn(train_mask, val_mask)
    print(f"  training time: {time.time() - t0:.0f}s")
    cnn_hist[direction] = hist

    xyz_te = torch.from_numpy(Xw[test_mask]).to(DEVICE)
    P = restrict(predict_probs(model, xyz_te, mu, sd))
    log_result(direction, "1D CNN (raw windows)", P, y_all[test_mask], meta[test_mask])
    del xyz_te
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 9. Results

In [ ]:
res = pd.DataFrame(results)
table = res.pivot_table(index="model", columns="direction", values=["window_acc", "vote_acc"])
table["mean window acc"] = res.groupby("model").window_acc.mean()
table["mean vote acc"] = res.groupby("model").vote_acc.mean()
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print(f"Chance level: {1 / len(EVAL_SUBJECTS):.3f}  |  leaky random split (reference only): {leaky:.3f}\n")
display(table.sort_values("mean window acc", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = res.groupby("model").window_acc.mean().sort_values(ascending=False).index
for ax, col, title in zip(axes, ["window_acc", "vote_acc"],
                          ["Window-level accuracy (cross-session)", f"Vote accuracy over {VOTE_K} windows (cross-session)"]):
    pv = res.pivot(index="model", columns="direction", values=col).loc[order]
    pv.plot(kind="bar", ax=ax)
    ax.axhline(1 / len(EVAL_SUBJECTS), color="gray", ls="--", lw=1, label="chance")
    ax.set_title(title); ax.set_ylim(0, 1); ax.set_ylabel("accuracy")
    ax.set_xticklabels(pv.index, rotation=30, ha="right")
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(cnn_hist), figsize=(6 * len(cnn_hist), 4), squeeze=False)
for ax, (d, h) in zip(axes[0], cnn_hist.items()):
    ax.plot(h["loss"], label="train loss")
    ax2 = ax.twinx()
    ax2.plot(h["val_acc"], color="tab:orange", label="val acc")
    ax.set_title(f"CNN training: {d}"); ax.set_xlabel("epoch")
    ax.set_ylabel("loss"); ax2.set_ylabel("val acc")
plt.tight_layout(); plt.show()

## 10. How to read these results

* **Window accuracy** is the realistic per-2-second identification rate across sessions. **Vote accuracy** shows what you get when you listen to a longer stretch of walking (`VOTE_K` windows).
* Compare the **invariant** and **full** feature sets: if "full" is clearly better, orientation carries identity information in this data; if it is worse, orientation shifts between sessions are hurting you.
* The **leaky random split** is far higher than every cross-session number. That gap is session/recording memorisation, not gait recognition. Always report the cross-session numbers.

**Ideas to improve further**
* Set `REMOVE_MEAN = False` and compare (gravity direction is orientation info).
* Increase `WIN` (e.g. 300-400) or lower `STEP` for more training windows.
* Try `ROT_DEG = 30-45` for stronger orientation augmentation.
* Ensemble the CNN with the tree models by averaging their probabilities.
* Metric learning (e.g. a triplet/ArcFace head) is a good next step if you need to identify subjects *not seen* in training.

## 10. Recurrent models: LSTM, CNN-LSTM, GRU, CNN-GRU

These models use the existing cross-session split, training-only normalization and augmentation, and the same window/vote metrics as the CNN and tree baselines. The recurrent models summarize the sequence with mean/max pooling; CNN hybrids first downsample local signal patterns to make recurrent training faster. Validation is from the training session and is only used to select a checkpoint, not as a substitute for cross-session test accuracy. Results can vary by hardware and random seed.

In [ ]:
class WindowRecurrentClassifier(nn.Module):
    def __init__(self, cell_type, use_cnn, n_classes):
        super().__init__()
        if use_cnn:
            self.frontend = nn.Sequential(
                conv_block(4, 32, 7), nn.MaxPool1d(2),
                conv_block(32, 64, 5), nn.MaxPool1d(2),
            )
            input_size = 64
        else:
            self.frontend = nn.Identity()
            input_size = 4

        recurrent = nn.LSTM if cell_type == "LSTM" else nn.GRU
        self.recurrent = recurrent(
            input_size=input_size,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, x):
        # x has shape (batch, channels, samples).
        sequence = self.frontend(x).transpose(1, 2)
        sequence, _ = self.recurrent(sequence)
        pooled = torch.cat([sequence.mean(dim=1), sequence.amax(dim=1)], dim=1)
        return self.head(pooled)


@torch.no_grad()
def predict_recurrent_probs(model, xyz_t, mu, sd, batch=1024):
    model.eval()
    outputs = []
    for i in range(0, len(xyz_t), batch):
        xb = (to_channels(xyz_t[i:i + batch]) - mu) / sd
        outputs.append(torch.softmax(model(xb), dim=1).cpu())
    return torch.cat(outputs).numpy()


def train_recurrent_model(model_name, cell_type, use_cnn, train_mask, val_mask,
                          epochs=16, batch_size=256, learning_rate=1e-3, patience=4):
    xyz_tr = torch.from_numpy(Xw[train_mask]).to(DEVICE)
    y_tr = torch.from_numpy(y_all[train_mask]).long().to(DEVICE)
    xyz_val = torch.from_numpy(Xw[val_mask]).to(DEVICE)
    y_val = y_all[val_mask]

    # Fit normalization using only the training portion of this direction.
    channels = to_channels(xyz_tr)
    mu = channels.mean(dim=(0, 2), keepdim=True)
    sd = channels.std(dim=(0, 2), keepdim=True).clamp_min(1e-6)
    del channels

    model = WindowRecurrentClassifier(cell_type, use_cnn, N_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(xyz_tr) / batch_size))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=learning_rate, epochs=epochs, steps_per_epoch=steps_per_epoch
    )
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_acc, best_state, epochs_without_improvement = -1.0, None, 0
    history = {"loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(len(xyz_tr), device=DEVICE)
        total_loss, seen = 0.0, 0
        for start in range(0, len(permutation), batch_size):
            indices = permutation[start:start + batch_size]
            xb = (to_channels(augment(xyz_tr[indices])) - mu) / sd
            loss = loss_fn(model(xb), y_tr[indices])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() * len(indices)
            seen += len(indices)

        val_probs = restrict(predict_recurrent_probs(model, xyz_val, mu, sd))
        val_acc = window_acc(val_probs, y_val)
        mean_loss = total_loss / seen
        history["loss"].append(mean_loss)
        history["val_acc"].append(val_acc)
        print(f"  {model_name} epoch {epoch + 1:02d}/{epochs} | loss {mean_loss:.4f} | val acc {val_acc:.2%}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"  Early stopping; best training-session validation accuracy: {best_acc:.2%}")
                break

    model.load_state_dict(best_state)
    del xyz_tr, y_tr, xyz_val, y_val, optimizer, scheduler
    return model, mu, sd, history


RECURRENT_MODELS = [
    ("LSTM", "LSTM", False),
    ("CNN-LSTM", "LSTM", True),
    ("GRU", "GRU", False),
    ("CNN-GRU", "GRU", True),
]
recurrent_model_names = {name for name, _, _ in RECURRENT_MODELS}
results[:] = [row for row in results if row["model"] not in recurrent_model_names]
recurrent_histories = {}

for train_session, test_session in DIRECTIONS:
    direction = f"S{train_session}->S{test_session}"
    train_mask, val_mask, test_mask = make_split(meta, train_session, test_session)
    print(f"\n=== Recurrent models {direction} | train {train_mask.sum():,} | val {val_mask.sum():,} | test {test_mask.sum():,} ===")
    xyz_test = torch.from_numpy(Xw[test_mask]).to(DEVICE)

    for model_name, cell_type, use_cnn in RECURRENT_MODELS:
        torch.manual_seed(SEED + train_session * 100 + len(recurrent_histories))
        started = time.time()
        model, mu, sd, history = train_recurrent_model(
            model_name, cell_type, use_cnn, train_mask, val_mask
        )
        probabilities = restrict(predict_recurrent_probs(model, xyz_test, mu, sd))
        log_result(direction, model_name, probabilities, y_all[test_mask], meta[test_mask])
        recurrent_histories[(direction, model_name)] = history
        print(f"  elapsed: {time.time() - started:.0f}s")
        del model, mu, sd, history, probabilities
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    del xyz_test

print("\nMean cross-session accuracy by model (both directions):")
recurrent_comparison = (pd.DataFrame(results)
                        .groupby("model")[["window_acc", "vote_acc"]]
                        .mean()
                        .sort_values("vote_acc", ascending=False))
display(recurrent_comparison.style.format("{:.2%}"))

## 11. Training-only tuning and subject-level evaluation

This experiment selects ExtraTrees hyperparameters using only the purged validation tail of the training session. It then refits the untuned baseline and selected model using all eligible recordings from the training session, and evaluates each once on the other session. Auxiliary single-session subjects are restricted to the training session by `make_split`. Hyperparameters are selected by macro-averaged subject vote accuracy (`VOTE_K=10`) so subjects with longer recordings do not dominate. The test table also reports fixed vote lengths of 5, 10, and 20 windows; these are comparisons, not settings selected from test results. Subject-bootstrap intervals describe variation across the 36 evaluation subjects, not uncertainty across independent sessions. With only two sessions, these results remain limited to this dataset.

**Important:** earlier cached cross-session scores are not valid under this corrected split: the old helper could include auxiliary-subject recordings from the held-out session. Those model/result outputs have been cleared. Restart the kernel and run the notebook from the top to reproduce corrected baseline and tuning results.

In [ ]:
def subject_vote_scores(P_eval, y_true, meta_part, k=VOTE_K):
    key = (meta_part.subject.astype(str).values + "_" +
           meta_part.session.astype(str).values + "_" +
           (meta_part.pos.values // k).astype(str))
    codes, _ = pd.factorize(key)
    grouped_probs = pd.DataFrame(P_eval).groupby(codes).mean().to_numpy()
    grouped_true = pd.Series(y_true).groupby(codes).first().to_numpy()
    grouped_subject = meta_part.reset_index(drop=True).groupby(codes).subject.first().to_numpy()
    predictions = EVAL_CLS[np.argmax(grouped_probs, axis=1)]
    return pd.Series((predictions == grouped_true).astype(float), index=grouped_subject).groupby(level=0).mean()


def bootstrap_subject_ci(subject_scores, rng, n_boot=2000):
    values = subject_scores.to_numpy(dtype=float)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return float(values.mean()), tuple(np.quantile(samples, [0.025, 0.975]))


EXTRA_TREES_CANDIDATES = [
    {"n_estimators": 200, "max_features": "sqrt", "min_samples_leaf": 1, "class_weight": None},
    {"n_estimators": 400, "max_features": "sqrt", "min_samples_leaf": 1, "class_weight": None},
    {"n_estimators": 400, "max_features": 0.50, "min_samples_leaf": 1, "class_weight": None},
    {"n_estimators": 400, "max_features": 0.25, "min_samples_leaf": 1, "class_weight": None},
    {"n_estimators": 400, "max_features": 0.50, "min_samples_leaf": 2, "class_weight": None},
    {"n_estimators": 400, "max_features": 0.25, "min_samples_leaf": 2, "class_weight": None},
    {"n_estimators": 400, "max_features": 0.50, "min_samples_leaf": 2, "class_weight": "balanced"},
    {"n_estimators": 400, "max_features": 0.25, "min_samples_leaf": 2, "class_weight": "balanced"},
]
TUNED_MODEL_NAME = "ExtraTrees tuned (invariant)"
BASELINE_MODEL_NAME = "ExtraTrees baseline (invariant, full train session)"
results[:] = [row for row in results if row["model"] not in {TUNED_MODEL_NAME, BASELINE_MODEL_NAME}]
tuning_tables, final_test_rows = [], []
subject_scores_by_model = {TUNED_MODEL_NAME: {}, BASELINE_MODEL_NAME: {}}
rng = np.random.default_rng(SEED)

for train_session, test_session in DIRECTIONS:
    direction = f"S{train_session}->S{test_session}"
    train_mask, val_mask, test_mask = make_split(meta, train_session, test_session)
    val_y = y_all[val_mask]
    val_meta = meta.loc[val_mask]
    print(f"\n=== Training-only tuning {direction} | train {train_mask.sum():,} | val {val_mask.sum():,} | test {test_mask.sum():,} ===")

    candidate_rows = []
    for candidate_id, params in enumerate(EXTRA_TREES_CANDIDATES, start=1):
        candidate = ExtraTreesClassifier(**params, n_jobs=N_JOBS, random_state=SEED)
        candidate.fit(F_inv[train_mask], y_all[train_mask])
        val_probs = restrict(to_full_probs(candidate.predict_proba(F_inv[val_mask]), candidate.classes_, N_CLASSES))
        score = subject_vote_scores(val_probs, val_y, val_meta, k=VOTE_K).mean()
        candidate_rows.append({"candidate": candidate_id, **params, "val_macro_vote_acc": float(score)})
        print(f"  candidate {candidate_id}: macro vote({VOTE_K}) {score:.2%} | {params}")

    tuning_table = pd.DataFrame(candidate_rows).sort_values(
        ["val_macro_vote_acc", "candidate"], ascending=[False, True]
    ).reset_index(drop=True)
    tuning_table["direction"] = direction
    tuning_tables.append(tuning_table)
    best_params = EXTRA_TREES_CANDIDATES[int(tuning_table.iloc[0]["candidate"]) - 1]
    print("  selected using validation only:", best_params)

    is_eval = meta.subject.isin(EVAL_SUBJECTS).values
    full_train_mask = (meta.session.values == train_session)
    if not USE_EXTRA_SUBJECTS:
        full_train_mask &= is_eval
    assert not np.any(full_train_mask & test_mask)
    assert (meta.session.values[full_train_mask] == train_session).all()

    test_y = y_all[test_mask]
    test_meta = meta.loc[test_mask]
    model_specs = [
        (BASELINE_MODEL_NAME, EXTRA_TREES_CANDIDATES[0]),
        (TUNED_MODEL_NAME, best_params),
    ]
    for model_name, params in model_specs:
        final_model = ExtraTreesClassifier(**params, n_jobs=N_JOBS, random_state=SEED)
        final_model.fit(F_inv[full_train_mask], y_all[full_train_mask])
        test_probs = restrict(to_full_probs(
            final_model.predict_proba(F_inv[test_mask]), final_model.classes_, N_CLASSES
        ))
        scores = subject_vote_scores(test_probs, test_y, test_meta, k=VOTE_K)
        mean_subject_acc, ci = bootstrap_subject_ci(scores, rng)
        row = {
            "direction": direction,
            "model": model_name,
            "window_acc": window_acc(test_probs, test_y),
            "vote5_acc": vote_acc(test_probs, test_y, test_meta, k=5),
            "vote10_acc": vote_acc(test_probs, test_y, test_meta, k=10),
            "vote20_acc": vote_acc(test_probs, test_y, test_meta, k=20),
            "macro_subject_vote10_acc": mean_subject_acc,
            "subject_bootstrap_ci_low": ci[0],
            "subject_bootstrap_ci_high": ci[1],
        }
        final_test_rows.append(row)
        subject_scores_by_model[model_name][direction] = scores
        log_result(direction, model_name, test_probs, test_y, test_meta)
        print(f"  macro subject vote(10): {mean_subject_acc:.2%} | subject-bootstrap 95% CI [{ci[0]:.2%}, {ci[1]:.2%}]")
        del final_model, test_probs

print("\nValidation-only hyperparameter rankings:")
display(pd.concat(tuning_tables, ignore_index=True).sort_values(
    ["direction", "val_macro_vote_acc"], ascending=[True, False]
).style.format({"val_macro_vote_acc": "{:.2%}"}))

final_test_table = pd.DataFrame(final_test_rows).set_index(["model", "direction"]).sort_index()
print("\nHeld-out session results (test settings were not used for selection):")
display(final_test_table.style.format("{:.2%}"))

print("\nPaired subject-bootstrap summary across both session directions:")
paired_summary = []
for model_name, by_direction in subject_scores_by_model.items():
    subject_ids = sorted(EVAL_SUBJECTS)
    values = np.stack([by_direction[f"S{tr}->S{te}"].reindex(subject_ids).to_numpy()
                       for tr, te in DIRECTIONS])
    sampled_subjects = rng.integers(0, len(subject_ids), size=(2000, len(subject_ids)))
    bootstrap_means = values[:, sampled_subjects].mean(axis=(0, 2))
    low, high = np.quantile(bootstrap_means, [0.025, 0.975])
    paired_summary.append({"model": model_name, "mean_macro_subject_vote10_acc": values.mean(),
                           "paired_subject_ci_low": low, "paired_subject_ci_high": high})
display(pd.DataFrame(paired_summary).set_index("model").style.format("{:.2%}"))